 ### 🎯 Module Overview
This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples.

Table of Contents

- Introduction to Data Ingestion
- Text Files (.txt)
- Markdown Files (.md)
- PDF Documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

## Introduction to Data Ingetion

In [1]:
import os
from typing import List, Dict, Any
import pandas as pd

In [2]:
from langchain_core.documents import Document
print("Successfully imported all the required modules.")

c:\Users\CHITTA\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully imported all the required modules.


### Understanding Document Structure in Langchain

In [9]:
## Create a simple document
doc = Document(
    page_content = "This is a sample document for testing the text splitters.", 
    metadata = {
        "source": "test.txt",
        "page": 1,
        "author": "Chittajit Nath",
        "date_created": "2026-08-10"
    }
)
print("Document created successfully.")

print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")
print()
print("📝 Metadata is crucial for:")
print("- Filtering Search Results.")
print("- Tracking the origin of the document.")
print("- Providing context in responses.")
print("- Debugging and auditing the system.")

Document created successfully.
Content: This is a sample document for testing the text splitters.
Metadata: {'source': 'test.txt', 'page': 1, 'author': 'Chittajit Nath', 'date_created': '2026-08-10'}

📝 Metadata is crucial for:
- Filtering Search Results.
- Tracking the origin of the document.
- Providing context in responses.
- Debugging and auditing the system.


In [4]:
type(doc)

langchain_core.documents.base.Document

## Text Files (.txt) - The Simplest Case {#2-text-files} 

In [5]:
os.makedirs("data/text_files", exist_ok = True)

In [6]:
sample_texts = {
    "data/text_files/python_intro.txt": "Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming.",
    "data/text_files/data_science.txt": "Data science is an interdisciplinary field that uses"
}

for file_path, content in sample_texts.items():
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"Created file: {file_path}")

Created file: data/text_files/python_intro.txt
Created file: data/text_files/data_science.txt


## Text Loader - Read Single File

In [6]:
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(f"Loaded {len(documents)} document(s) from the text file.")
print(f"Content of the first document:\n{documents[0].page_content[:200]}...")  # Print the first 100 characters
print(f"Metadata of the first document:\n{documents[0].metadata}")

Loaded 1 document(s) from the text file.
Content of the first document:
What Is Python
Python is a high-level programming language used in software and web development. Developers use Python to build backend logic, automate tasks, and process data. In web projects, Python...
Metadata of the first document:
{'source': 'data/text_files/python_intro.txt'}


## Directory Loader

### Advantages:
- Loades multiple files
- Supports glob patterns
- Progress tracking 
- Recursive Directory Scanning

### Disadvantages:
- All files must be same type
- Limited error handeling per file
- Can be memory intesive for large directories

In [17]:
## DirectoryLoader - Multiple Text files
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
    "data/text_files", 
    glob = "**/*.txt", 
    loader_cls = TextLoader,
    loader_kwargs = {"encoding": "utf-8"}, 
    show_progress=True)

documents = dir_loader.load()
print(f"Loaded {len(documents)} document(s) from the directory.")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"Content: {doc.page_content[:100]}...")  # Print the first 100 characters
    print(f"Source: {doc.metadata['source']}")
    print(f"Length of content: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 3202.98it/s]

Loaded 2 document(s) from the directory.

Document 1:
Content: Data science is an interdisciplinary field that uses...
Source: data\text_files\data_science.txt
Length of content: 52 characters

Document 2:
Content: What Is Python
Python is a high-level programming language used in software and web development. Dev...
Source: data\text_files\python_intro.txt
Length of content: 3111 characters


## Text Splitting Statergies

In [19]:
## Different text splitting strategies
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print(documents)

[Document(metadata={'source': 'data\\text_files\\data_science.txt'}, page_content='Data science is an interdisciplinary field that uses'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='What Is Python\nPython is a high-level programming language used in software and web development. Developers use Python to build backend logic, automate tasks, and process data. In web projects, Python usually runs on the server and supports the application layer behind the user interface. Python is a general-purpose programming language. Therefore, it is not limited to one use case. Developers use it for web applications, automation workflows, testing, system administration, and data-driven tools. In backend development, Python handles key tasks such as request processing, validation, and communication with databases.  It is also known for clear syntax and readability. This matters as readable code is easier to review, maintain, and extend. Furthermore, Python has a b

### Character Text Splitter

In [22]:
text = documents[1].page_content
text

'What Is Python\nPython is a high-level programming language used in software and web development. Developers use Python to build backend logic, automate tasks, and process data. In web projects, Python usually runs on the server and supports the application layer behind the user interface. Python is a general-purpose programming language. Therefore, it is not limited to one use case. Developers use it for web applications, automation workflows, testing, system administration, and data-driven tools. In backend development, Python handles key tasks such as request processing, validation, and communication with databases.  It is also known for clear syntax and readability. This matters as readable code is easier to review, maintain, and extend. Furthermore, Python has a broad ecosystem of libraries and frameworks. These tools help developers implement standard features without writing every component from the beginning. \n\nWhy Python Matters in Web Development\nPython remains relevant i

In [25]:
# Method 1: Character-based splitting
print("1️⃣ Character Text Splittier")
char_splitter = CharacterTextSplitter(
    separator = "\n",       # Splis on newline
    chunk_size = 200,       # Max characters per chunk 
    chunk_overlap = 20,     # Overlap between chunks
    length_function = len   # Function to measure length of text
)

char_chunks = char_splitter.split_text(text)
print(f"Number of chunks created: {len(char_chunks)}")
print(f"Chunks: {char_chunks[0]}")
print(f"Chunks: {char_chunks[1]}")

Created a chunk of size 915, which is longer than the specified 200
Created a chunk of size 310, which is longer than the specified 200
Created a chunk of size 225, which is longer than the specified 200
Created a chunk of size 272, which is longer than the specified 200
Created a chunk of size 267, which is longer than the specified 200
Created a chunk of size 210, which is longer than the specified 200
Created a chunk of size 333, which is longer than the specified 200


1️⃣ Character Text Splittier
Number of chunks created: 14
Chunks: What Is Python
Chunks: Python is a high-level programming language used in software and web development. Developers use Python to build backend logic, automate tasks, and process data. In web projects, Python usually runs on the server and supports the application layer behind the user interface. Python is a general-purpose programming language. Therefore, it is not limited to one use case. Developers use it for web applications, automation workflows, testing, system administration, and data-driven tools. In backend development, Python handles key tasks such as request processing, validation, and communication with databases.  It is also known for clear syntax and readability. This matters as readable code is easier to review, maintain, and extend. Furthermore, Python has a broad ecosystem of libraries and frameworks. These tools help developers implement standard features without writing every component from the beginni

In [27]:
# Method 2: Recursive Character-based splitting (Recommended)
print("2️⃣ Recursive Character Text Splittier")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],  # Hierarchy of separators
    chunk_size = 200,                      # Max characters per chunk
    chunk_overlap = 20,                    # Overlap between chunks
    length_function = len                  # Function to measure length of text
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Number of chunks created: {len(recursive_chunks)}")
print(f"Chunks: {recursive_chunks[0]}")
print(f"Chunks: {recursive_chunks[1]}")
print(f"Chunks: {recursive_chunks[2]}")

2️⃣ Recursive Character Text Splittier
Number of chunks created: 25
Chunks: What Is Python
Chunks: Python is a high-level programming language used in software and web development. Developers use Python to build backend logic, automate tasks, and process data. In web projects, Python usually runs
Chunks: Python usually runs on the server and supports the application layer behind the user interface. Python is a general-purpose programming language. Therefore, it is not limited to one use case.


In [28]:
# Method 3: Token-based splitting
print("3️⃣ Token Text Splittier")
token_splitter = TokenTextSplitter(
    chunk_size = 50,       # Max tokens per chunk
    chunk_overlap = 20     # Overlap between chunks
)

token_chunks = token_splitter.split_text(text)
print(f"Number of chunks created: {len(token_chunks)}")
print(f"Chunks: {token_chunks[0]}")
print(f"Chunks: {token_chunks[1]}")

3️⃣ Token Text Splittier
Number of chunks created: 19
Chunks: What Is Python
Python is a high-level programming language used in software and web development. Developers use Python to build backend logic, automate tasks, and process data. In web projects, Python usually runs on the server and supports the application layer behind
Chunks:  and process data. In web projects, Python usually runs on the server and supports the application layer behind the user interface. Python is a general-purpose programming language. Therefore, it is not limited to one use case. Developers use it for web applications


## Text Splitting Model Comparison

| CharacterTextSplitter | RecursiveCharacterTextSplitter | TokenTextSplitter |
| --- | --- | ---|
| ✅ Simple and predictable | ✅ Respects text structure | ✅ Respects model token limit |
| ✅ Good for structured text | ✅ Tries multiple separators | ✅ More accurate for embeddings |
|  | ✅ Best General purpose splitter |  |
| ❌ May break mid-sentence | ❌ Slightly more complex | ❌ Slower than character-based | 
| Use: Text has clear delimiters | Use: Default choice for most texts | Use: Working with token limited models |